In [2]:
import requests
import json

MAST_URL = "https://mast.stsci.edu/api/v0/invoke"


def mast_query(payload):
    response = requests.post(MAST_URL, data={"request": json.dumps(payload)})
    result = response.json()

    # Robust error handling
    if "status" in result and result["status"] == "error":
        raise Exception(f"MAST API Error: {result}")

    if "data" not in result:
        raise Exception(f"Unexpected response: {result}")

    return result["data"]


# -------------------------------
# STEP 1: Get JWST observations
# -------------------------------
obs_query = {
    "service": "Mast.Caom.Filtered",
    "params": {
        "columns": "obsid,instrument_name,target_name",
        "filters": [
            {"paramName": "obs_collection", "values": ["JWST"]}
        ]
    },
    "format": "json"
}

obs_data = mast_query(obs_query)

print(f"Total JWST observations: {len(obs_data)}")

# -------------------------------
# STEP 2: Filter instruments (correct names!)
# -------------------------------
filtered_obs = [
    o for o in obs_data
    if o["instrument_name"] in ["NIRCam", "MIRI"]
]

print(f"NIRCam + MIRI observations: {len(filtered_obs)}")

# Take a few obsids (avoid huge downloads)
obsids = [o["obsid"] for o in filtered_obs[:3]]

# -------------------------------
# STEP 3: Fetch products
# -------------------------------
all_uncal = []

for obsid in obsids:
    prod_query = {
        "service": "Mast.Caom.Products",
        "params": {
            "obsid": obsid,
            "columns": "*"
        },
        "format": "json"
    }

    products = mast_query(prod_query)

    # -------------------------------
    # STEP 4: Filter RAW (uncalibrated)
    # -------------------------------
    uncal = [
        p for p in products
        if p.get("calib_level") == 1
    ]

    print(f"obsid {obsid} → {len(uncal)} uncal files")

    all_uncal.extend(uncal)

# -------------------------------
# STEP 5: Show download URLs
# -------------------------------
print("\nSample uncal files:\n")

for f in all_uncal[:5]:
    print(f["dataURI"])

Total JWST observations: 408808
NIRCam + MIRI observations: 0

Sample uncal files:



In [3]:
print(set(o["instrument_name"] for o in obs_data[:100]))

{'NIRSPEC/MSA', 'NIRISS/IMAGE', 'NIRCAM/GRISM', 'NIRISS/WFSS', 'NIRSPEC', 'NIRCAM/IMAGE', 'NIRCAM/CORON', 'NIRSPEC/IFU', 'MIRI/IMAGE', 'MIRI/IFU', 'NIRSPEC/SLIT'}


In [4]:
filtered_obs = [
    o for o in obs_data
    if "NIRCAM" in o["instrument_name"] or "MIRI" in o["instrument_name"]
]

In [5]:
# STEP 2: Filter instruments (FIXED)

filtered_obs = [
    o for o in obs_data
    if ("NIRCAM" in o["instrument_name"]) or ("MIRI" in o["instrument_name"])
]

print(f"NIRCam + MIRI observations: {len(filtered_obs)}")

NIRCam + MIRI observations: 180565


In [6]:
filtered_obs = [
    o for o in obs_data
    if ("NIRCAM" in o["instrument_name"].upper()) or
       ("MIRI" in o["instrument_name"].upper())
]

## Retrieveing the data

In [7]:
def mast_url(uri):
    return f"https://mast.stsci.edu/api/v0.1/Download/file?uri={uri}"

In [8]:
import os
import requests

os.makedirs("data", exist_ok=True)

def download_file(uri):
    url = f"https://mast.stsci.edu/api/v0.1/Download/file?uri={uri}"
    filename = uri.split("/")[-1]
    filepath = os.path.join("data", filename)

    print(f"⬇️ Downloading {filename}...")

    r = requests.get(url, stream=True)
    with open(filepath, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

    print(f"✅ Saved to {filepath}")

In [9]:
for f in all_uncal[:10]:   # only 3 files first!
    download_file(f["dataURI"])

In [10]:
import os
print(os.listdir("data"))

[]


In [11]:
from astropy.io import fits

hdul = fits.open("data/your_file_uncal.fits")
hdul.info()

FileNotFoundError: [Errno 2] No such file or directory: 'data/your_file_uncal.fits'

In [ ]:
uncal = [
    p for p in products
    if str(p.get("calib_level")) == "1"
    and "uncal" in p.get("productFilename", "").lower()
]

NameError: name 'products' is not defined